In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Define directories
INPUT_DIR = "output/02-shirorekha"
OUTPUT_DIR = "output/03-segmentation"
CHARACTER_DIRS = os.path.join(OUTPUT_DIR, "characters")

# Create directories if they don't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
for zone in ["upper", "middle", "lower"]:
    os.makedirs(os.path.join(CHARACTER_DIRS, zone), exist_ok=True)

# Input image
image_filename = "shirorekha-removed-preprocessed-0008.png"
input_path = os.path.join(INPUT_DIR, image_filename)

In [ ]:
# Helper function to display images
def show_image(image, title="Image", cmap_type="gray"):
    plt.figure(figsize=(6, 4))
    plt.imshow(image, cmap=cmap_type)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# -------------------- Load and Binarize (Text = 1, BG = 0) --------------------
original_img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
binary = (original_img == 0).astype("uint8")

In [ ]:
# -------------------- Zone Estimation --------------------
num_labels_init, _, stats_init, _ = cv2.connectedComponentsWithStats(
    binary, connectivity=8
)

heights = stats_init[1:, cv2.CC_STAT_HEIGHT]
centers_y = stats_init[1:, cv2.CC_STAT_TOP] + (heights / 2)

median_height = np.median(heights)
median_center = np.median(centers_y)

# Boundaries for Upper, Middle, and Lower zones
middle_top = int(median_center - 0.55 * median_height)
middle_bottom = int(median_center + 0.55 * median_height)

# Visualize zones
zones = cv2.cvtColor(original_img, cv2.COLOR_GRAY2BGR)
h_img, w_img = original_img.shape

# Draw boundaries
cv2.rectangle(zones, (0, 0), (w_img - 1, middle_top), (255, 0, 0), 1)
cv2.rectangle(zones, (0, middle_top), (w_img - 1, middle_bottom), (0, 0, 255), 1)
cv2.rectangle(zones, (0, middle_bottom), (w_img - 1, h_img - 1), (0, 255, 0), 1)

# Display
show_image(zones, title="Estimated Zones")

zones_save_path = os.path.join(OUTPUT_DIR, "01-zones-" + image_filename)
cv2.imwrite(zones_save_path, zones)
print(f"Zone visualization saved to: {zones_save_path}")

In [ ]:
# -------------------- Split Attached Lower Modifiers --------------------
split_binary = binary.copy()
search_margin = int(median_height * 0.20)

for i in range(1, num_labels_init):
    x, y, w, h = stats_init[i, :4]
    bottom_y = y + h

    # Component crosses into the lower zone
    if y < middle_bottom and bottom_y > (middle_bottom + search_margin):
        roi = split_binary[y:bottom_y, x : x + w]

        # Search window
        local_middle = middle_bottom - y
        search_start = max(0, local_middle - search_margin)
        search_end = min(roi.shape[0], local_middle + search_margin)

        if search_start < search_end:
            row_sums = np.sum(roi[search_start:search_end, :], axis=1)
            if len(row_sums) > 0:
                local_split_y = search_start + np.argmin(row_sums)
                global_split_y = y + local_split_y
                split_binary[max(0, global_split_y - 1) : global_split_y, x : x + w] = 0

In [ ]:
# -------------------- Fill horizontal gaps in Lower zone --------------------
zone_top = middle_bottom
zone_bottom = split_binary.shape[0]
image_width = split_binary.shape[1]
max_gap = 4

for r in range(zone_top, zone_bottom):
    c = 0
    while c < image_width:
        if split_binary[r, c] == 1:
            c2 = c + 1
            gap = 0

            while c2 < image_width and split_binary[r, c2] == 0:
                gap += 1
                if gap > max_gap:
                    break
                c2 += 1

            if c2 < image_width and gap > 0 and gap <= max_gap:
                split_binary[r, c + 1 : c2] = 1
                c = c2
            else:
                c += 1
        else:
            c += 1

# Display
show_image(split_binary, title="Split Binary")

#  Save
split_binary_save_path = os.path.join(OUTPUT_DIR, f"02-lower-split-{image_filename}")
cv2.imwrite(split_binary_save_path, split_binary * 255)
print(f"Saved: {split_binary_save_path}")

In [ ]:
# -------------------- Component Extraction --------------------
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
    split_binary, connectivity=8
)

categorized_components = {"upper": [], "middle": [], "lower": []}

for i in range(1, num_labels):
    x, y, w, h = stats[i, :4]
    cy = y + (h / 2)

    if cy < middle_top:
        zone = "upper"
    elif cy > middle_bottom:
        zone = "lower"
    else:
        zone = "middle"

    crop = original_img[y - 1 : y + h, x - 1 : x + w]
    padded_crop = cv2.copyMakeBorder(crop, 4, 4, 4, 4, cv2.BORDER_CONSTANT, value=255)

    categorized_components[zone].append(
        {"x": x, "y": y, "w": w, "h": h, "crop": padded_crop}
    )

In [ ]:
# -------------------- Character Segmentation --------------------
bbox_img = cv2.cvtColor(split_binary * 255, cv2.COLOR_GRAY2BGR)

for zone in categorized_components:
    categorized_components[zone].sort(key=lambda item: item["x"])
    for idx, comp in enumerate(categorized_components[zone]):
        cx, cy, cw, ch, cropped = (
            comp["x"],
            comp["y"],
            comp["w"],
            comp["h"],
            comp["crop"],
        )
        cv2.rectangle(bbox_img, (cx, cy), (cx + cw, cy + ch), (255, 255, 0), 1)

        fname = f"{idx:02d}_x{cx}.png"
        cv2.imwrite(os.path.join(CHARACTER_DIRS, zone, fname), cropped)

bbox_save_path = os.path.join(OUTPUT_DIR, f"03-bbox-{image_filename}")
cv2.imwrite(bbox_save_path, bbox_img)
print(f"Saved: {bbox_save_path}")

show_image(bbox_img, title="Bounding Boxes")